# CoT Distillation v2 — Kaggle Pipeline

Reproduces Magister et al. (ACL 2023) at small scale on FLAN-T5-base, plus
ReCEval (Prasad et al., EMNLP 2023) as a second evaluation axis.

**This notebook is the v2 successor to `cot-gpt.ipynb`.** v1 ran end-to-end
but produced post-distillation accuracy *below* the zero-shot baseline
(see `doc/Current Notebook.md` in the repo). v2 fixes the recipe (lr 3e-4 →
5e-5, weight_decay 0 → 0.01, epochs 3 → 8 with early stopping, max_seq
256 → 512) and the decoding (greedy → beam=4 + no_repeat_ngram=4 +
repetition_penalty=1.15). It also adds two conditions (Direct FT and a
calculator-corrected filter, Set C).

**Authoritative spec:** `AGENT.md` in the repo.

## How to use this notebook

1. Run the **Setup** cells once per session to clone/pull and install deps.
2. Run the **Config** cell to confirm the v2 recipe; tweak inline if you
   want to A/B a hyperparameter (it patches `config/config.yaml` in place).
3. Run stages in order. Each stage cell prints a `python -m src.status`
   summary at the end so you can see where the project stands without
   scrolling.
4. The four real Stage-3 runs are split into separate cells so you can
   run them one at a time across Kaggle sessions, resuming with `--resume`.
5. **Persist outputs:** Kaggle's `/kaggle/working` survives only the current
   session. The final cell zips and uploads `outputs/checkpoints/`,
   `outputs/generations/`, and `outputs/runs/` to a Kaggle Dataset so you
   can pick up across sessions.

## 0 · Setup

In [ ]:
# 0.1 — Clone OR pull the repo. Idempotent: run every session.
# Hard-resets the workdir to origin/<BRANCH> so stale local edits in the
# Kaggle ephemeral workspace (e.g. exec-bit churn on shell scripts) can't
# block the pull.
import os, subprocess, pathlib

REPO_URL  = "https://github.com/rihembenabdallah18/COT_lab.git"
BRANCH    = "dev"      # switch to 'main' once v2 is merged
REPO_NAME = "COT_lab"
WORKDIR   = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle").exists() else pathlib.Path.cwd()
REPO_DIR  = WORKDIR / REPO_NAME

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    # Discard any stale local edits in the ephemeral Kaggle workdir before resetting.
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

In [ ]:
# 0.2 — Install dependencies.
# torch is preinstalled on Kaggle; we install the rest from requirements.txt.
# `peft` if present can shadow imports — uninstall to be safe.
!pip install -q -r requirements.txt
!pip uninstall -y -q peft || true
!python -m spacy download en_core_web_sm -q

In [ ]:
# 0.3 — GPU check. Should print True + Tesla T4 (or P100 on Kaggle premium).
import torch
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("mem (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: CUDA not available - training will be unusably slow on CPU. "
          "Switch the Kaggle accelerator to GPU (Settings -> Accelerator).")

## 1 · Config — review and (optionally) override

The single source of truth is `config/config.yaml`. The cell below loads it,
shows the v2 recipe, and lets you patch any field for an ablation without
editing the file by hand. Set `OVERRIDES = {}` to keep the committed v2
values (recommended for the first run).

In [ ]:
import yaml, pathlib, json

CFG_PATH = pathlib.Path("config/config.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())

# ---- Edit here for ablations ----------------------------------------------
OVERRIDES = {
    # "learning_rate": 3.0e-5,
    # "num_epochs": 6,
    # "max_input_length": 384,
    # "max_target_length": 384,
    # "inference_num_beams": 5,
}
# ---------------------------------------------------------------------------

for k, v in OVERRIDES.items():
    if k not in cfg:
        raise KeyError(f"Unknown config key: {k}")
    cfg[k] = v

if OVERRIDES:
    CFG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("applied overrides:", OVERRIDES)

print(json.dumps({k: cfg[k] for k in [
    "model_name", "learning_rate", "weight_decay", "warmup_ratio",
    "num_epochs", "early_stopping_patience", "max_input_length",
    "max_target_length", "batch_size", "gradient_accumulation_steps",
    "fp16", "inference_num_beams", "inference_no_repeat_ngram_size",
    "inference_repetition_penalty", "inference_max_new_tokens", "seed"
]}, indent=2))

## 2 · Project status (run any time)

Reads `outputs/runs/*.json` and prints one row per stage run. Use this
between cells to see what's done.

In [ ]:
!python -m src.status

## 3 · Stage 1 — Download GSM8K + Ho et al. teacher CoTs

Idempotent: skips files already on disk. Reports the Ho et al. JSON schema
and prints 5 example train rows + their teacher CoT.

In [ ]:
!bash scripts/01_download.sh

## 4 · Stage 2 — Build the four training sets

Produces `direct_ft.jsonl`, `set_a_nofilter.jsonl`, `set_b_magister.jsonl`,
`set_c_calculator.jsonl`. Set C is the calculator-corrected, *process-aware*
filter — see `AGENT.md §4`. Writes a Stage-2 run-card.

In [ ]:
!bash scripts/02_filter.sh
!python -m src.status

### 4.1 — Unit tests

27 cases covering the answer parser and the calculator. Run before any
training to catch regressions if you bumped a library version.

In [ ]:
!python -m pytest tests/ -q

## 5 · Stage 3 — Fine-tune the four students

Run the cells **in order**. Each is independent and resumable
(`--resume` picks up from the latest checkpoint after a session timeout).

Per AGENT.md, run cheapest-first so the recipe is validated before the
longest run:

1. **Smoke** (200 ex, 1 epoch) — verifies the loop end-to-end
2. **Direct FT** — Ho et al. 4.93% reference; cheapest real run
3. **Set B** (Magister filter, 3,389)
4. **Set C** (calculator-corrected, 2,635)
5. **Set A** (no filter, 7,473) — longest

After Direct FT finishes, **stop and inspect the run-card.** If `best_eval_loss`
is well above ~1.0 the recipe is still wrong — escalate before the longer runs.

### 5.0 — Storage cleanup helper

Each finished training run leaves two `checkpoint-N/` dirs (best + most-recent
under `save_total_limit=2`). Each contains `optimizer.pt` (~750 MB for AdamW
on FLAN-T5-base) plus scheduler/RNG state — pure training-resume state that
inference doesn't need. The helper below:

1. Reads `trainer_state.best_model_checkpoint` to find the best-by-eval-loss
   checkpoint (which is *not necessarily* the highest-step one).
2. Deletes every non-best checkpoint dir.
3. Strips `optimizer.pt`, `scheduler.pt`, `rng_state*.pth` from the survivor.

This typically frees ~1.5 GB per run while preserving everything Stage 4
inference + the run-card need.

In [ ]:
import json, pathlib, shutil

CKPT_ROOT = pathlib.Path("outputs/checkpoints")
_TRAIN_STATE_FILES = ("optimizer.pt", "scheduler.pt")
_RNG_GLOB = "rng_state*.pth"

def _dir_size(p: pathlib.Path) -> int:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def _fmt(b: int) -> str:
    return f"{b/1e9:.2f} GB" if b >= 1e9 else f"{b/1e6:.0f} MB"

def cleanup_run(run_name: str, keep_best_only: bool = True) -> None:
    """Strip post-training waste from outputs/checkpoints/<run_name>/.

    Always deletes optimizer.pt, scheduler.pt, rng_state*.pth from the
    surviving checkpoint(s). When keep_best_only=True (default) also deletes
    every checkpoint dir that isn't the best by eval_loss.
    """
    run_dir = CKPT_ROOT / run_name
    if not run_dir.exists():
        print(f"[cleanup:{run_name}] run dir not found, skipping.")
        return

    ckpts = sorted(run_dir.glob("checkpoint-*"),
                   key=lambda p: int(p.name.split("-")[-1]))
    if not ckpts:
        print(f"[cleanup:{run_name}] no checkpoint-* dirs found.")
        return

    before = _dir_size(run_dir)

    # Identify the best checkpoint via the most recent trainer_state.json.
    best_name = None
    state_path = ckpts[-1] / "trainer_state.json"
    if state_path.exists():
        bmc = json.loads(state_path.read_text()).get("best_model_checkpoint")
        if bmc:
            best_name = pathlib.Path(bmc).name  # e.g. "checkpoint-1234"

    kept = []
    for ckpt in ckpts:
        if keep_best_only and best_name and ckpt.name != best_name:
            shutil.rmtree(ckpt)
            print(f"[cleanup:{run_name}] removed non-best {ckpt.name}")
            continue
        for fname in _TRAIN_STATE_FILES:
            f = ckpt / fname
            if f.exists():
                f.unlink()
        for f in ckpt.glob(_RNG_GLOB):
            f.unlink()
        kept.append(ckpt.name)

    after = _dir_size(run_dir)
    print(f"[cleanup:{run_name}] kept {kept} | "
          f"freed {_fmt(before - after)} ({_fmt(before)} -> {_fmt(after)})")

### 5.1 — Smoke (200 ex, 1 epoch, ~10 min on T4)

In [ ]:
!bash scripts/03_train_smoke.sh

### 5.2 — Direct FT (Q→A only, no CoT)

Recipe-validation gate. After this finishes, check the run-card.

In [ ]:
!bash scripts/03_train_direct_ft.sh
!python -m src.status

In [ ]:
cleanup_run("student_direct_ft")

### 5.3 — Set B (Magister filter)

In [ ]:
!bash scripts/03_train_set_b.sh
!python -m src.status

In [ ]:
cleanup_run("student_set_b")

### 5.4 — Set C (calculator-corrected, process-aware)

In [ ]:
!bash scripts/03_train_set_c.sh
!python -m src.status

In [ ]:
cleanup_run("student_set_c")

### 5.5 — Set A (no filter, longest run)

In [ ]:
!bash scripts/03_train_set_a.sh
!python -m src.status

In [ ]:
cleanup_run("student_set_a")

## 6 · Stage 4 — Inference on the GSM8K test set

Runs all five conditions with v2 decoding (beam=4, no_repeat_ngram=4,
repetition_penalty=1.15). Each writes a JSONL and a run-card. Resumable.

In [ ]:
!bash scripts/04_inference.sh
!python -m src.status

## 7 · Stage 5 — Evaluation

Stage 5 splits into accuracy (5a) and ReCEval (5b). The cells are
commented-out until those scripts land in the repo; uncomment after pulling
a commit that adds them.

In [ ]:
# 7.1 — Accuracy + accuracy-with-calculator (Stage 5a)
# !bash scripts/05a_accuracy.sh
# !cat outputs/eval_results/accuracy.csv

In [ ]:
# 7.2 — ReCEval (Stage 5b)
# !bash scripts/05b_receval.sh
# !cat outputs/eval_results/receval_summary.csv

## 8 · Persist outputs across sessions

Kaggle wipes `/kaggle/working` between sessions for free notebooks. The cell
below copies `outputs/checkpoints/`, `outputs/generations/`, `outputs/runs/`,
`outputs/eval_results/`, and `data/processed/` into a staging dir and pushes
them to a Kaggle Dataset.

**Auth is automatic when this notebook runs inside Kaggle** — `KAGGLE_USERNAME`
and `KAGGLE_KEY` are already in the env, so the `kaggle` CLI just works. No
secrets to set up. Set `BACKUP_DATASET_SLUG` below if you want a name other
than the default.

In [ ]:
# 8.1 — Package outputs/ + data/processed/ and push to a Kaggle Dataset.
# Auto-detects the Kaggle username from the env when running inside Kaggle.
# Outside Kaggle (e.g. local codespace), this is a no-op.
import json, os, pathlib, shutil, subprocess

ON_KAGGLE = pathlib.Path("/kaggle").exists()
KAGGLE_USERNAME     = os.environ.get("KAGGLE_USERNAME", "")
BACKUP_DATASET_SLUG = "cot-lab-v2-outputs"

if not ON_KAGGLE:
    print("not on Kaggle - skipping dataset upload.")
elif not KAGGLE_USERNAME:
    print("KAGGLE_USERNAME not set in env - skipping. "
          "(On Kaggle this is normally injected automatically.)")
else:
    STAGE_DIR = pathlib.Path("/kaggle/working/_dataset_payload")
    STAGE_DIR.mkdir(parents=True, exist_ok=True)
    for src in ["outputs/checkpoints", "outputs/generations", "outputs/runs",
                "outputs/eval_results", "data/processed"]:
        src_path = pathlib.Path(src)
        if src_path.exists():
            dst = STAGE_DIR / src_path.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(src_path, dst)
            print("staged:", src, "->", dst)

    meta = {
        "title":    "COT_lab v2 outputs",
        "id":       f"{KAGGLE_USERNAME}/{BACKUP_DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    (STAGE_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    # Try update first; fall back to create if the dataset doesn't exist yet.
    res = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(STAGE_DIR),
         "-m", "v2 outputs update", "-r", "zip"],
        capture_output=True, text=True,
    )
    if res.returncode != 0 and "Not Found" in (res.stderr + res.stdout):
        res = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(STAGE_DIR), "-r", "zip"],
            capture_output=True, text=True,
        )
    print(res.stdout)
    if res.returncode != 0:
        print("STDERR:", res.stderr)

### 8.2 — Fallback: zip outputs for manual download

If you don't want (or can't) push a Kaggle Dataset, this packs everything
into a single `outputs.tar.gz` you can download from the Kaggle file browser
(or `cp` from a local codespace).

In [ ]:
import pathlib, subprocess

ARCHIVE = pathlib.Path("/kaggle/working/cot_lab_outputs.tar.gz") \
    if pathlib.Path("/kaggle").exists() else pathlib.Path("cot_lab_outputs.tar.gz")

paths = [p for p in ["outputs/checkpoints", "outputs/generations",
                      "outputs/runs", "outputs/eval_results",
                      "data/processed"] if pathlib.Path(p).exists()]
if not paths:
    print("nothing to archive yet.")
else:
    subprocess.run(["tar", "-czf", str(ARCHIVE), *paths], check=True)
    size_mb = ARCHIVE.stat().st_size / 1e6
    print(f"wrote {ARCHIVE} ({size_mb:.1f} MB)")
    print("contents:")
    for pp in paths:
        print(" -", pp)